# Training YOLO to automate labeling process

### Environment Setup

In [10]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import torch
import albumentations as A
import numpy as np
import shutil
from pathlib import Path
import glob
import yaml

# Windows paths - TODO: Update these paths to your actual dataset folders
train_folder = r"C:\Users\tfgmo\iCloudDrive\Mahjong CV\train_dataset"      # Training images and labels
test_folder = r"C:\Users\tfgmo\iCloudDrive\Mahjong CV\test_dataset"        # Test images and labels  
val_folder = r"C:\Users\tfgmo\iCloudDrive\Mahjong CV\val_dataset"          # Validation images and labels
output_folder = r"C:\Users\tfgmo\Documents\GitHub\Mahjong-Recognition\notebooks\data\processed"  # Output organized dataset

# Use the standalone data.yaml file
data_path = r"C:\Users\tfgmo\Documents\GitHub\Mahjong-Recognition\notebooks\data.yaml"

In [2]:
# def create_yolo_config(output_folder, class_names=None):
#     """
#     Create YOLO dataset configuration file (data.yaml)
    
#     Args:
#         output_folder: Path to organized dataset
#         class_names: List of class names. If None, will attempt to infer from labels
#     """
    
#     # If class names not provided, try to infer from label files
#     if class_names is None:
#         class_ids = set()
        
#         # Check all label files to find class IDs
#         for split in ['train', 'val', 'test']:
#             labels_dir = f"{output_folder}/{split}/labels"
#             if os.path.exists(labels_dir):
#                 for label_file in os.listdir(labels_dir):
#                     if label_file.endswith('.txt'):
#                         label_path = os.path.join(labels_dir, label_file)
#                         with open(label_path, 'r') as f:
#                             for line in f:
#                                 if line.strip():
#                                     class_id = int(line.strip().split()[0])
#                                     class_ids.add(class_id)
        
#         # Create default class names
#         max_class_id = max(class_ids) if class_ids else 0
#         class_names = [f"class_{i}" for i in range(max_class_id + 1)]
#         print(f"Inferred {len(class_names)} classes from labels: {class_names}")
    
#     # Create configuration
#     config = {
#         'path': os.path.abspath(output_folder),
#         'train': 'train/images',
#         'val': 'val/images',
#         'test': 'test/images',
#         'nc': len(class_names),
#         'names': class_names
#     }
    
#     # Write configuration file
#     config_path = f"{output_folder}/data.yaml"
#     with open(config_path, 'w') as f:
#         yaml.dump(config, f, default_flow_style=False)
    
#     print(f"Created YOLO config file: {config_path}")
#     print(f"Configuration: {config}")
    
#     return config_path

# # Create the YOLO configuration file
# # TODO: Update class_names if you know the specific classes in your dataset
# config_path = create_yolo_config(output_folder, class_names=None)

In [11]:
def create_class_mapping():
    """
    Create mapping from original class IDs (with UNKNOWN) to new class IDs (without UNKNOWN).
    
    Original classes (38 total):
    0-33: valid classes
    34: UNKNOWN (to be removed)
    35-37: valid classes
    
    New classes (37 total):
    0-33: same as original
    34-36: shifted from original 35-37
    """
    mapping = {}
    new_id = 0
    
    # Map original class IDs to new class IDs
    for original_id in range(38):  # 0 to 37
        if original_id == 34:  # Skip UNKNOWN
            continue
        mapping[original_id] = new_id
        new_id += 1
    
    return mapping

def filter_and_remap_labels(label_content, class_mapping, unknown_class_id=34):
    """
    Filter out UNKNOWN class labels and remap remaining class IDs.
    
    Args:
        label_content: List of label lines
        class_mapping: Dictionary mapping original class IDs to new class IDs
        unknown_class_id: Class ID for UNKNOWN class (default 34)
    
    Returns:
        Filtered and remapped label content
    """
    filtered_lines = []
    for line in label_content:
        if line.strip():
            parts = line.strip().split()
            original_class_id = int(parts[0])
            
            # Skip UNKNOWN class
            if original_class_id == unknown_class_id:
                continue
            
            # Remap class ID
            if original_class_id in class_mapping:
                new_class_id = class_mapping[original_class_id]
                parts[0] = str(new_class_id)
                filtered_lines.append(' '.join(parts) + '\n')
            else:
                print(f"Warning: Unknown class ID {original_class_id} found")
    
    return filtered_lines

def prepare_yolo_dataset_from_folders(train_folder, test_folder, val_folder, output_folder):
    """
    Prepare YOLO dataset by organizing images and labels from separate train/test/val folders.
    Filters out UNKNOWN class labels and remaps class IDs during preparation.
    Works with Windows paths.
    
    Args:
        train_folder: Path to training data folder (images + txt labels)
        test_folder: Path to test data folder (images + txt labels)
        val_folder: Path to validation data folder (images + txt labels)
        output_folder: Path to output organized dataset
    """
    
    # Create class mapping
    class_mapping = create_class_mapping()
    print(f"Class mapping created: {class_mapping}")
    
    # Create output directory structure
    for split in ['train', 'val', 'test']:
        for data_type in ['images', 'labels']:
            split_dir = os.path.join(output_folder, split, data_type)
            os.makedirs(split_dir, exist_ok=True)
    
    # Mapping of input folders to output splits
    folder_mapping = {
        'train': train_folder,
        'test': test_folder,
        'val': val_folder
    }
    
    # Process each folder
    for split_name, input_folder in folder_mapping.items():
        if not os.path.exists(input_folder):
            print(f"Warning: {split_name} folder not found: {input_folder}")
            continue
            
        # Find all image files using Windows-compatible glob
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.JPG', '*.JPEG', '*.PNG']
        image_files = []
        for ext in image_extensions:
            pattern = os.path.join(input_folder, ext)
            image_files.extend(glob.glob(pattern))
        
        print(f"Found {len(image_files)} images in {input_folder}")
        
        processed_count = 0
        skipped_count = 0
        
        # Copy images and corresponding labels
        for img_path in image_files:
            filename = os.path.basename(img_path)
            name_without_ext = os.path.splitext(filename)[0]
            
            # Check corresponding label file
            label_filename = name_without_ext + '.txt'
            src_label = os.path.join(input_folder, label_filename)
            
            if os.path.exists(src_label):
                # Read and filter/remap label content
                with open(src_label, 'r', encoding='utf-8') as f:
                    label_lines = f.readlines()
                
                # Filter out UNKNOWN class and remap class IDs
                filtered_lines = filter_and_remap_labels(label_lines, class_mapping, unknown_class_id=34)
                
                # Skip images that have no valid labels after filtering
                if not filtered_lines:
                    print(f"Skipping {filename} - only contains UNKNOWN labels")
                    skipped_count += 1
                    continue
                
                # Copy image
                dst_img = os.path.join(output_folder, split_name, 'images', filename)
                shutil.copy2(img_path, dst_img)
                
                # Write filtered and remapped label file
                dst_label = os.path.join(output_folder, split_name, 'labels', label_filename)
                with open(dst_label, 'w', encoding='utf-8') as f:
                    f.writelines(filtered_lines)
                
                processed_count += 1
            else:
                print(f"Warning: Label file not found for {filename}")
                skipped_count += 1
        
        print(f"Split {split_name}: processed {processed_count} images, skipped {skipped_count} images")
    
    print("Dataset preparation completed!")

# Prepare the dataset from separate folders
prepare_yolo_dataset_from_folders(train_folder, test_folder, val_folder, output_folder)

Class mapping created: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 35: 34, 36: 35, 37: 36}
Dataset preparation completed!


# Remove the old config creation code since we now use the standalone data.yaml file
print(f"Using data configuration file: {data_path}")
print("Make sure your data.yaml file paths are correct for your Windows system!")

# 加载 YOLO 模型
model = YOLO('yolo11l.pt')  # 使用预训练的 YOLO11l 模型 

# 开始训练 - 使用standalone data.yaml文件
results = model.train(
    data=data_path,            # 使用standalone data.yaml文件
    # imgsz=640,                 # 图片大小
    epochs=500,                 # 训练轮数
    batch=10,                  # 批量大小
    name='mahjong-yolo-custom', # 训练结果保存的项目名称
    pretrained=True,            # 是否使用预训练权重
    device='cuda',
    lr0=0.01,              # 初始学习率，视情况调整
    patience=20,
    half=True,
    cache='ram',
    workers=8,
)

In [ ]:
# 加载 YOLO 模型
model = YOLO('yolo12s.pt')  # 使用预训练的 YOLO11x 模型 

# 开始训练
results = model.train(
    data=data_path,  # 数据集配置文件路径
    # imgsz=640,                 # 图片大小
    epochs=500,                 # 训练轮数
    batch=16,                  # 批量大小
    name='mahjong-yolo_12s',       # 训练结果保存的项目名称
    pretrained=True,            # 是否使用预训练权重
    device='cuda',
    lr0=0.001,              # 初始学习率，视情况调整
    patience=20,
    half=True,
    cache='ram',
    workers=8,
)


Ultralytics 8.3.167  Python-3.11.9 torch-2.9.0.dev20250715+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\tfgmo\Documents\GitHub\Mahjong-Recognition\notebooks\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=mahjong-yolo_12s2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

train: Scanning C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final.cache... 2340 images, 320 backgrounds, 382 corrupt: 100%|██████████| 2340/2340 [00:00<?, ?it/s]

train: C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final\000013_0.png: ignoring corrupt image/label: Label class 37 exceeds dataset class count 37. Possible class labels are 0-36
train: C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final\000013_1.png: ignoring corrupt image/label: Label class 37 exceeds dataset class count 37. Possible class labels are 0-36
train: C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final\000013_10.png: ignoring corrupt image/label: Label class 37 exceeds dataset class count 37. Possible class labels are 0-36
train: C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final\000013_11.png: ignoring corrupt image/label: Label class 37 exceeds dataset class count 37. Possible class labels are 0-36
train: C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\train_data_final\000013_12.png: ignoring corrupt image/label: Label class 37 exceeds dataset clas

WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (1.9GB RAM): 100%|██████████| 1958/1958 [00:13<00:00, 141.92it/s]

...

WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


val: Caching images (0.0GB RAM): 100%|██████████| 2/2 [00:00<00:00,  9.01it/s]


Plotting labels to c:\Users\tfgmo\Documents\GitHub\Mahjong-Recognition\runs\detect\mahjong-yolo_12s2\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 113 weight(decay=0.0), 120 weight(decay=0.0005), 119 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to c:\Users\tfgmo\Documents\GitHub\Mahjong-Recognition\runs\detect\mahjong-yolo_12s2
Starting training for 500 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/500        30G      2.139      4.267       1.68       1297        640:  84%|████████▍ | 52/62 [41:04<07:54, 47.40s/it]


KeyboardInterrupt: 

# 调整边界框宽度和字体大小
def draw_boxes_with_custom_style(image, boxes, labels, line_thickness=0.1, font_scale=0.1, font_thickness=0.1):
    """
    在图片上绘制边界框，支持自定义边框宽度和字体大小。

    :param image: 原始图片
    :param boxes: 边界框信息
    :param labels: 类别标签
    :param line_thickness: 边界框线条宽度
    :param font_scale: 字体大小
    :param font_thickness: 字体粗细
    :return: 绘制边界框后的图片
    """
    for box in boxes:
        cls = int(box.cls)  # 类别索引
        conf = float(box.conf)  # 置信度
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # 边界框坐标

        # 绘制边界框
        cv2.rectangle(image, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=line_thickness)

        # 绘制标签
        label = f"{labels[cls]}: {conf:.2f}"
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
        label_x = x1
        label_y = y1 - 10 if y1 - 10 > 10 else y1 + 10
        cv2.rectangle(image, (label_x, label_y - label_size[1]), (label_x + label_size[0], label_y), (0, 255, 0), -1)
        cv2.putText(image, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness)
    
    return image

# 从测试集中选择一张图片进行预测
test_images_dir = os.path.join(output_folder, "test", "images")
if os.path.exists(test_images_dir):
    test_images = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    if test_images:
        # 使用第一张测试图片
        image_path = os.path.join(test_images_dir, test_images[0])
        print(f"Testing with image: {image_path}")
        
        # 进行预测
        results = model.predict(source=image_path, conf=0.25)

        # 获取检测结果
        boxes = results[0].boxes  # 检测到的边界框
        labels = results[0].names  # 类别标签
        image = cv2.imread(image_path)

        # 绘制边界框，调整线宽和字体大小
        image_with_custom_boxes = draw_boxes_with_custom_style(
            image.copy(), boxes, labels, line_thickness=1, font_scale=0.8, font_thickness=1
        )

        # 保存结果到文件（可选）
        output_path = "detection_results.png"
        cv2.imwrite(output_path, image_with_custom_boxes)

        # 显示检测结果图片
        plt.figure(figsize=(12, 12))
        plt.imshow(cv2.cvtColor(image_with_custom_boxes, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("Detection Results with Custom Style")
        plt.show()
    else:
        print("No test images found in test folder.")
else:
    print("Test images directory not found. Please run the data preparation cell first.")

In [6]:
# 调整边界框宽度和字体大小
def draw_boxes_with_custom_style(image, boxes, labels, line_thickness=0.1, font_scale=0.1, font_thickness=0.1):
    """
    在图片上绘制边界框，支持自定义边框宽度和字体大小。

    :param image: 原始图片
    :param boxes: 边界框信息
    :param labels: 类别标签
    :param line_thickness: 边界框线条宽度
    :param font_scale: 字体大小
    :param font_thickness: 字体粗细
    :return: 绘制边界框后的图片
    """
    for box in boxes:
        cls = int(box.cls)  # 类别索引
        conf = float(box.conf)  # 置信度
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # 边界框坐标

        # 绘制边界框
        cv2.rectangle(image, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=line_thickness)

        # 绘制标签
        label = f"{labels[cls]}: {conf:.2f}"
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
        label_x = x1
        label_y = y1 - 10 if y1 - 10 > 10 else y1 + 10
        cv2.rectangle(image, (label_x, label_y - label_size[1]), (label_x + label_size[0], label_y), (0, 255, 0), -1)
        cv2.putText(image, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness)
    
    return image

# TODO: Update this path to your test image location
image_path = r'C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\test_data_final\validate\000009.png'

# 进行预测
results = model.predict(source=image_path, conf=0.25)

# 获取检测结果
boxes = results[0].boxes  # 检测到的边界框
labels = results[0].names  # 类别标签
image = cv2.imread(image_path)

# 绘制边界框，调整线宽和字体大小
image_with_custom_boxes = draw_boxes_with_custom_style(
    image.copy(), boxes, labels, line_thickness=1, font_scale=0.8, font_thickness=1
)

# 保存结果到文件（可选）
output_path = "detection_results.png"
cv2.imwrite(output_path, image_with_custom_boxes)

# 显示检测结果图片
plt.figure(figsize=(12, 12))
plt.imshow(cv2.cvtColor(image_with_custom_boxes, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Detection Results with Custom Style")
plt.show()


image 1/1 C:\Users\tfgmo\iCloudDrive\Mahjong CV\augmented_dataset_final\test_data_final\validate\000009.png: 480x640 2 1ms, 3 1ps, 5 1zs, 5 2ps, 2 2ss, 5 2zs, 1 3p, 3 3ss, 3 3zs, 4 4ps, 1 4s, 3 4zs, 4 5zs, 1 6p, 5 6ss, 1 6z, 2 7ps, 3 7ss, 4 7zs, 1 8m, 1 8s, 3 9ms, 2 9ps, 1 9s, 1 0s, 22.6ms
Speed: 2.8ms preprocess, 22.6ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)


<Figure size 1200x1200 with 1 Axes>